In [1]:
import pandas as pd
import math
import glob
import os
import torch
import random
from torch.nn.utils.rnn import pad_sequence

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [2]:
files = sorted(glob.glob("COMMENTARY_INTL_MATCH/*.csv"))

df = pd.concat(
    [pd.read_csv(f, usecols=['Commentary_short', 'Commentary']) for f in files if os.path.isfile(f)],
    ignore_index=True
)
df

,Commentary,Commentary_short
0,"on the pads to start from Amir, no swing, work...","Mohammad Amir to Renshaw, 1 run"
1,"drifts down leg this time, no swing whatsoever...","Mohammad Amir to Warner, no run"
2,off the outside half and that races away. Bett...,"Mohammad Amir to Warner, FOUR runs"
3,"leg side-ish again, 140 kph. Amir looking for ...","Mohammad Amir to Warner, 1 leg bye"
4,"much better. 139 kph, coming back in on off, p...","Mohammad Amir to Renshaw, no run"
...,...,...
735686,"low full on middle, Baby goes low and scoops i...","Kumar to Sachin Baby, 2 runs"
735687,"rip-roaring yorker shading outside leg, the ba...","Kumar to Sachin Baby, OUT"
735688,"low full toss on middle, there is the war cry ...","Kumar to Iqbal Abdulla, 1 leg bye"
735689,"full and outside off, backs away and carves it...","Kumar to Sachin Baby, 1 run"


In [3]:
data = df.rename(columns={
    'Commentary_short': 'input',
    'Commentary': 'output'
})
data = data[['input','output']]
data

,input,output
0,"Mohammad Amir to Renshaw, 1 run","on the pads to start from Amir, no swing, work..."
1,"Mohammad Amir to Warner, no run","drifts down leg this time, no swing whatsoever..."
2,"Mohammad Amir to Warner, FOUR runs",off the outside half and that races away. Bett...
3,"Mohammad Amir to Warner, 1 leg bye","leg side-ish again, 140 kph. Amir looking for ..."
4,"Mohammad Amir to Renshaw, no run","much better. 139 kph, coming back in on off, p..."
...,...,...
735686,"Kumar to Sachin Baby, 2 runs","low full on middle, Baby goes low and scoops i..."
735687,"Kumar to Sachin Baby, OUT","rip-roaring yorker shading outside leg, the ba..."
735688,"Kumar to Iqbal Abdulla, 1 leg bye","low full toss on middle, there is the war cry ..."
735689,"Kumar to Sachin Baby, 1 run","full and outside off, backs away and carves it..."


In [4]:
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
data = data.dropna(subset=['input', 'output'])
data

,input,output
1,"Dananjaya to Taylor, no run","full on leg stump, pushed towards short midwic..."
2,"Mohammad Abbas to Hope, no run","floats this full outside off, this swung away ..."
3,"Ngidi to Lakmal, no run",blocks a fuller one into the covers
4,"Ahmed to Finch, 2 runs","length ball on the pads, tucks it away towards..."
5,"Rashid Khan to Balbirnie, 1 run","too full to do anything, pushed off the front ..."
...,...,...
735686,"Dananjaya to Mohammad Saifuddin, no run","full on leg, blocked"
735687,"Perera to Patterson, 1 leg bye","full on leg stump, and he goes down on one kne..."
735688,"Harbhajan Singh to Mishra, 1 run",bunts a quicker delivery on off stump down to ...
735689,"Nortje to Bhanuka, no run",defended from a length


In [5]:
# data.to_csv("commentary_checks.csv")

In [6]:
data['text'] = "[BOS] " + data['input'] + " => " + data['output'] + " [EOS]"
text = data['text'].tolist()

In [7]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size = 5000,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

tokenizer.train_from_iterator(text, trainer=trainer)

In [8]:
PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")

def encode(s):
    return tokenizer.encode(s).ids

def decode(ids):
    return tokenizer.decode(ids)

print(encode("[BOS] Virat     Kohli get's his 100! Brilliant century from him. [EOS]"))
print(decode([2, 2014, 2675, 108, 108, 108, 108, 564, 284, 247, 242, 3220, 4, 4781, 4160, 280, 361, 17, 108, 3]))

[2, 2014, 2675, 108, 108, 108, 108, 564, 284, 247, 242, 3220, 4, 4781, 4160, 280, 361, 17, 108, 3]
 Virat     Kohli get's his 100! Brilliant century from him. 


In [9]:
encoded_data = [encode(t) for t in text]
print(encode("=>"))

[130]


In [ ]:
# print(encode("[PAD]"))
# print(encode("[UNK]"))
# print(encode("[BOS]"))
# print(encode("[EOS]"))

[0]
[1]
[2]
[3]


In [10]:
n1 = int(0.8 * len(encoded_data))
n2 = int(0.9 * len(encoded_data))
train_data = encoded_data[:n2]
val_data = encoded_data[n1:n2]
test_data = encoded_data[n2:]

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def get_batch(split, batch_size=32):
    if split == 'train':
        data = train_data
    elif split == 'val':
        data = val_data
    elif split == 'test':
        data = test_data
    else:
        print("Invalid split input. Please try again!")
        return
    
    idx = random.sample(range(len(data)), batch_size)
    
    x = [torch.tensor(data[i][:-1]) for i in idx]
    y = [torch.tensor(data[i][1:]) for i in idx]
    
    x = pad_sequence(x, batch_first=True, padding_value=0)
    y = pad_sequence(y, batch_first=True, padding_value=0)

    x = x.to(device)
    y = y.to(device)
    
    return x, y

Using device: cuda


In [ ]:
# check = get_batch('train')

In [ ]:
# ix = 10
# print(decode(check[0][ix].tolist())) #decodes will show the same for [0][ix] and [1][ix] because decode ignores BOS and EOS
# print(check[0][ix])
# print(decode(check[1][ix].tolist()))
# print(check[1][ix])

 Morris to Rahane, 1 run => full ball, and he makes room but can only get this down towards long off 
tensor([   2, 1360,  115,  829,   15,  174,  131,  130,  222,  204,   15,  138,
         203, 1001,  726,  236,  545, 1024,  284,  238,  252,  331,  333,  150,
         108,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
       device='cuda:0')
 Morris to Rahane, 1 run => full ball, and he makes room but can only get this down towards long off 
tensor([1360,  115,  829,   15,  174,  131,  130,  222,  204,   15,  138,  203,
        1001,  726,  236,  545, 1024,  284,  238,  252,  331,  333,  150,  108,
           3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,  

In [12]:
print("Max token id:", max(max(seq) for seq in encoded_data))
print("Vocab size:", tokenizer.get_vocab_size())
print("Max number of tokens in one line:", max(len(seq) for seq in encoded_data))

Max token id: 4999
Vocab size: 5000
Max number of tokens in one line: 308


In [13]:
class position_encoding:

    def __init__(self,emb_dim,device):
        self.matrix = torch.zeros(308,emb_dim,device=device)
        for j in range(308):
            for i in range(emb_dim):
                if i%2 == 0:
                    self.matrix[j][i] = math.sin(j/(10000**(2*(i//2)/emb_dim)))
                else:
                    self.matrix[j][i] = math.cos(j/(10000**(2*(i//2)/emb_dim)))
    
    def forward(self, input_tensor):
        #input_tensor is B,T,E
        T = input_tensor.shape[1] #changes for every batch cuz of padding
        return input_tensor + 0.05*self.matrix[:T].unsqueeze(0)

In [14]:
class self_attention:

    def __init__(self, emb_dim, heads, device):
        self.heads = heads
        self.emb_dim = emb_dim
        self.head_dim = int(self.emb_dim/self.heads)
        g = torch.Generator(device=device).manual_seed(47364748)
        self.W_q = (torch.randn(self.heads, self.emb_dim, self.head_dim, generator=g, device=device) * 0.02).requires_grad_()
        self.W_k = (torch.randn(self.heads, self.emb_dim, self.head_dim, generator=g, device=device) * 0.02).requires_grad_()
        self.W_v = (torch.randn(self.heads, self.emb_dim, self.head_dim, generator=g, device=device) * 0.02).requires_grad_()

    def __call__(self,x):
        b,t,e = x.shape
        x = x.unsqueeze(1).expand(b,self.heads,t,e) #b,h,t,e
        W_q = self.W_q.unsqueeze(0) #1,h,e,d
        W_k = self.W_k.unsqueeze(0)
        W_v = self.W_v.unsqueeze(0)
        query = x@W_q #b,h,t,d
        value = x@W_v
        key = x@W_k
        QK = query@key.permute(0,1,3,2)
        masked = torch.tril(torch.ones_like(QK))
        QK = QK.masked_fill(masked==0,float('-inf'))
        QK = QK/(self.head_dim**0.5)
        # QK = QK - QK.max(3, keepdim=True).values
        # QK = QK.exp()
        # QK = QK/QK.sum(3,keepdim=True)
        QK = torch.softmax(QK, dim=3)
        y_faux = QK@value
        out = y_faux.permute(0,2,1,3)
        out = out.contiguous().view(b,t,e)
        
        return out
    
    def parameters(self):
        return [self.W_q, self.W_v, self.W_k]

In [15]:
class feed_forward_network:

    def __init__(self, emb_dim, device):
        g = torch.Generator(device=device).manual_seed(47364748)
        self.W1 = (torch.randn(emb_dim,4*emb_dim,generator=g,device=device) * 0.02).requires_grad_()
        self.b1 = torch.zeros(4*emb_dim,device=device).requires_grad_()
        self.W2 = (torch.randn(4*emb_dim, emb_dim,generator=g,device=device) * 0.02).requires_grad_()
        self.b2 = torch.zeros(emb_dim,device=device).requires_grad_()

    def __call__(self,x):
        x = x @ self.W1 + self.b1
        x = torch.relu(x)
        x = x @ self.W2 + self.b2
        return x
    
    def parameters(self):
        return [self.W1, self.b1, self.W2, self.b2]

In [16]:
class layer_norm:
    
    def __init__(self, emb_dim, device):
        self.gamma = torch.ones(emb_dim,device=device).requires_grad_()
        self.beta  = torch.zeros(emb_dim,device=device).requires_grad_()
        self.eps = 1e-5
    
    def __call__(self,x):
        mean = x.mean(2,keepdim=True)
        var  = x.var(2,keepdim=True,unbiased=False)
        x_hat = (x - mean)/torch.sqrt(var + self.eps) #(B,T,E)

        return self.gamma*x_hat + self.beta #multiplies each embedding dimension with it's unique learnt gamma and beta
    
    def parameters(self):
        return [self.gamma, self.beta]

In [ ]:
# F = torch.tensor([[[4,5,6,7], 
#                    [6,7,3,2]],
#                   [[7,8,4,2],
#                    [9,4,7,1]],
#                   [[2,3,6,5],
#                    [6,8,9,1]]],dtype=float)
# F.shape

torch.Size([3, 2, 4])

In [ ]:
# print(F.mean(2))
# print(F.mean(2,keepdim=True))

tensor([[5.5000, 4.5000],
        [5.2500, 5.2500],
        [4.0000, 6.0000]], dtype=torch.float64)
tensor([[[5.5000],
         [4.5000]],

        [[5.2500],
         [5.2500]],

        [[4.0000],
         [6.0000]]], dtype=torch.float64)


In [ ]:
# F = (F - F.mean(2,keepdim=True))/F.std(2,keepdim=True)
# F

tensor([[[-1.1619, -0.3873,  0.3873,  1.1619],
         [ 0.6301,  1.0502, -0.6301, -1.0502]],

        [[ 0.6355,  0.9986, -0.4539, -1.1802],
         [ 1.0714, -0.3571,  0.5000, -1.2143]],

        [[-1.0954, -0.5477,  1.0954,  0.5477],
         [ 0.0000,  0.5620,  0.8429, -1.4049]]], dtype=torch.float64)

In [17]:
class block:

    def __init__(self, emb_dim, heads, device):
        self.attn = self_attention(emb_dim,heads,device)
        self.ln1 = layer_norm(emb_dim,device=device)
        self.ffn = feed_forward_network(emb_dim,device)
        self.ln2 = layer_norm(emb_dim,device=device)
    
    def __call__(self,x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
    
    def parameters(self):
        return self.attn.parameters() + self.ffn.parameters() + self.ln1.parameters() + self.ln2.parameters()

In [18]:
g = torch.Generator(device=device).manual_seed(2147483647)
emb_dim = 200
vocab_size = 5000
heads = 10
head_dim = int(emb_dim/heads)
layers = 3

C = (torch.randn(vocab_size,emb_dim,generator=g,device=device) * 0.02).requires_grad_()
W_vocab = (torch.randn(emb_dim,vocab_size,generator=g,device=device) * 0.02).requires_grad_()
pe = position_encoding(emb_dim,device)
model = [block(emb_dim,heads,device) for _ in range(layers)]
params = [C, W_vocab]
for layer in model:
    params += layer.parameters()

In [19]:
for step in range(3000):
    X,Y = get_batch("train",128)
    emb = C[X]
    emb = pe.forward(emb)
    x = emb
    for layer in model:
        x = layer(x)
    pred_y = x
    # print(emb.shape)
    # print(pred_y.shape)
    logits = pred_y @ W_vocab #(32 or batchsize,emb.shape[1] or no. of tokens,vocab_size)
    # print(logits.shape)
    B,T,V = logits.shape
    loss = torch.nn.functional.cross_entropy(logits.view(B*T, V), Y.view(B*T),ignore_index=PAD_ID)

    for p in params:
        p.grad = None
    loss.backward()

    lr = 0.01 if step < 5000 else 0.001
    for p in params:
        p.data -= lr*p.grad

    if step%500 == 0:
        print(step,loss.item())

0 8.559553146362305
500 5.387937545776367
1000 5.204636096954346
1500 4.87582540512085
2000 4.688573360443115
2500 4.573601722717285


In [28]:
-torch.log(torch.tensor(1/5000))

tensor(8.5172)

In [ ]:
X,Y = get_batch("val", 128)
emb = C[X]
emb = pe.forward(emb)
x = emb
for layer in model:
    x = layer(x)
pred_y = x
# print(emb.shape)
# print(pred_y.shape)
logits = pred_y @ W_vocab #(32 or batchsize,emb.shape[1] or no. of tokens,vocab_size)
# print(logits.shape)
B,T,V = logits.shape
loss = torch.nn.functional.cross_entropy(logits.view(B*T, V), Y.view(B*T),ignore_index=PAD_ID)
print(loss.item())

#after just self attention it is around 20
#after adding positon encoding it was around 18
#after ffn and before layernorm it is 5.95

4.352553844451904


In [21]:
C[X].shape

torch.Size([128, 109, 200])

In [22]:
def generate_text(input, max_new_tokens=308):
    # model.eval()
    tokens = encode(input)
    x = torch.tensor(tokens,dtype=torch.long,device=device).unsqueeze(0) #(1,T)
    for _ in range(max_new_tokens):
        x_cond = x[:, -308:]
        emb = C[x_cond] #(1,T,emb_dim)
        emb = pe.forward(emb)
        x_1 = emb
        for layer in model:
            x_1 = layer(x_1)
        pred_y = x_1
        logits = pred_y @ W_vocab #(1,T,vocab_size)
        last_logits = logits[0,-1,:] #(vocab_size)
        temperature = 0.7
        k = 50
        topk_vals, topk_idx = torch.topk(last_logits,k)
        probs = torch.softmax(topk_vals/temperature,dim=0)
        next_token = topk_idx[torch.multinomial(probs,num_samples=1)].item()
        x = torch.cat([x,torch.tensor([[next_token]],device=device)],dim=1)
        if next_token == encode("[EOS]")[0]:
            break
    return decode(x[0].tolist())

In [26]:
generate_text("[BOS] Amir to Kholi, 3 runs =>")

' Amir to Kholi, 3 runs => full outside the on off, good short and a heon to that he through in into the the it on the back.> '